# Lab 4 — Proximity Detector (LOF)

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

> **Quick check:** precision ≈ **0.33** · recall (fraud) = **1.00** · train legit **792** rows




## Why this matters <!-- cisco-doc-enrich-2026 -->

Train only on **legitimate** transactions; large distance to "normal" = anomaly. Classic approach
when labels are scarce.

```
legit cloud  (train)          new point far away --> flag
        o o o o o
      o o o o o o
```


## Semi-supervised anomaly detection

Train on **legitimate** transactions only; label **-1** = outlier.

In [1]:
from pathlib import Path

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

# cisco-output-ready
print(f"Setup OK — repo root: {GH_ROOT}")


Setup OK — repo root: C:\25-Trainings\2-Confirmed\10-June-26-AI-ML-Cisco\GH


**Step 2** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Continue** — next code block. <!-- cisco-doc-enrich-2026 -->


In [3]:
# cisco-debug-summary
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print("dtypes:", df.dtypes.to_dict())
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Positive rate — train: {y_train.mean():.2%} | test: {y_test.mean():.2%}")

Loaded 1,000 rows x 6 columns
dtypes: {'transaction_id': dtype('O'), 'timestamp': dtype('O'), 'amount': dtype('float64'), 'merchant_category': dtype('O'), 'distance_from_home': dtype('float64'), 'is_fraud': dtype('int64')}
Train: 800 | Test: 200
Positive rate — train: 1.00% | test: 1.00%


**Step 3** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [4]:
X_train_legit = X_train[y_train == 0]
print(f"train total: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"train legit only: {len(X_train_legit)}")
print(f"test: {len(X_test)} (fraud {int(y_test.sum())})")

train total: 800 (fraud 8)
train legit only: 792
test: 200 (fraud 2)


## Preprocess and fit LOF on legit train only

In [5]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

X_train_legit_s = preprocess.fit_transform(X_train_legit)
X_test_s = preprocess.transform(X_test)

CONTAMINATION = 0.02
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
lof.fit(X_train_legit_s)

pred = lof.predict(X_test_s)
y_pred = np.where(pred == -1, 1, 0)

# cisco-debug-summary
print("Model fit complete.")

Model fit complete.


**Step 2** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [6]:
print("Lab 4 — Proximity detector (LOF)")
print(f"train legit rows: {len(X_train_legit)}")
print(f"test rows: {len(X_test)}")
print(f"predicted anomalies: {int((y_pred == 1).sum())}")

Lab 4 — Proximity detector (LOF)
train legit rows: 792
test rows: 200
predicted anomalies: 6


## Precision and recall (fraud)

In [7]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"precision (fraud): {precision:.4f}")
print(f"recall (fraud): {recall:.4f}")

results = pd.DataFrame({
    "metric": ["precision (fraud)", "recall (fraud)"],
    "value": [precision, recall],
})
display(results.round(4))


precision (fraud): 0.3333
recall (fraud): 1.0000


,metric,value
0,precision (fraud),0.3333
1,recall (fraud),1.0000


## Prediction breakdown

In [8]:
breakdown = pd.DataFrame({
    "actual_fraud": y_test.values,
    "predicted_fraud": y_pred,
    "amount": X_test["amount"].values,
    "distance_from_home": X_test["distance_from_home"].values,
})
display(breakdown.sort_values("predicted_fraud", ascending=False).head(10).round(2))


,actual_fraud,predicted_fraud,amount,distance_from_home
69,0,1,323.82,0.06
128,0,1,93.40,0.36
81,0,1,102.88,0.99
170,0,1,87.22,9.50
163,1,1,510.55,27.68
73,1,1,45.73,38.33
2,0,0,13.10,4.64
0,0,0,5.92,0.48
7,0,0,22.47,2.57
8,0,0,42.74,9.37


## Extension — vary contamination

In [9]:
rows = []
for contam in [0.01, 0.02, 0.05]:
    model = LocalOutlierFactor(n_neighbors=20, contamination=contam, novelty=True)
    model.fit(X_train_legit_s)
    yp = np.where(model.predict(X_test_s) == -1, 1, 0)
    rows.append({
        "contamination": contam,
        "predicted_anomalies": int((yp == 1).sum()),
        "precision": precision_score(y_test, yp, zero_division=0),
        "recall": recall_score(y_test, yp, zero_division=0),
    })

display(pd.DataFrame(rows).round(4))


# cisco-debug-summary
print("Model fit complete.")

,contamination,predicted_anomalies,precision,recall
0,0.01,4,0.5000,1.0
1,0.02,6,0.3333,1.0
2,0.05,12,0.1667,1.0


Model fit complete.


### LOF prompt 1

Show LOF raw predict labels.

In [10]:
print(np.unique(pred, return_counts=True))

(array([-1,  1]), array([  6, 194]))


### LOF prompt 2

Count true positives.

In [11]:
print(int(((y_pred==1)&(y_test==1)).sum()))

2


### LOF prompt 3

Count false positives.

In [12]:
print(int(((y_pred==1)&(y_test==0)).sum()))

4


### LOF prompt 4

Count false negatives.

In [13]:
print(int(((y_pred==0)&(y_test==1)).sum()))

0


### LOF prompt 5

Display contamination used.

In [14]:
print(CONTAMINATION)

0.02


### LOF prompt 6

Show n_neighbors setting.

In [15]:
print(lof.n_neighbors)

20


### LOF prompt 7

List predicted anomaly amounts.

In [16]:
print(breakdown.loc[breakdown['predicted_fraud']==1, 'amount'].round(2).tolist())

[323.82, 45.73, 102.88, 93.4, 510.55, 87.22]


### LOF prompt 8

Compare train legit vs total train.

In [17]:
print(len(X_train_legit), len(X_train))

792 800


### LOF prompt 9

Verify novelty mode.

In [18]:
print(lof.novelty)

True


### LOF prompt 10

Show test fraud rows in breakdown.

In [19]:
display(breakdown.loc[breakdown['actual_fraud']==1].round(2))

,actual_fraud,predicted_fraud,amount,distance_from_home
73,1,1,45.73,38.33
163,1,1,510.55,27.68


### LOF prompt 11

Compute anomaly rate on test.

In [20]:
print(round((y_pred==1).mean(), 4))

0.03


### LOF prompt 12

Re-run precision manually.

In [21]:
print(precision_score(y_test, y_pred, zero_division=0))

0.3333333333333333

### LOF prompt 13

Re-run recall manually.

In [22]:
print(recall_score(y_test, y_pred, zero_division=0))

1.0


### LOF prompt 14

Summarize LOF mapping.

In [23]:
print('LOF -1 mapped to fraud prediction 1')

LOF -1 mapped to fraud prediction 1

### LOF prompt 15

Show feature matrix shapes.

In [24]:
print(X_train_legit_s.shape, X_test_s.shape)

(792, 9) (200, 9)


### LOF prompt 16

State semi-supervised assumption.

In [25]:
print('Train density on legit-only; score all test rows.')

Train density on legit-only; score all test rows.


### LOF prompt 17

Compare to Day 5 DBSCAN noise idea.

In [26]:
print('Both flag sparse/outlier points in feature space.')

Both flag sparse/outlier points in feature space.


### LOF prompt 18

Display top distances among flagged.

In [27]:
display(breakdown.loc[breakdown['predicted_fraud']==1].nlargest(3, 'distance_from_home'))

,actual_fraud,predicted_fraud,amount,distance_from_home
73,1,1,45.727218,38.33
163,1,1,510.551672,27.68
170,0,1,87.220000,9.50


### LOF prompt 19

Reconfirm test size.

In [28]:
print(len(X_test))

200


### LOF prompt 20

Bridge to Random Forest.

In [29]:
print('Lab 5 uses supervised RF with class_weight balanced.')

Lab 5 uses supervised RF with class_weight balanced.


### Lab 4 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [30]:
print("Lab 4 recap step 1: completed")

Lab 4 recap step 1: completed


### Lab 4 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [31]:
print("Lab 4 recap step 2: completed")

Lab 4 recap step 2: completed


### Lab 4 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [32]:
print("Lab 4 recap step 3: completed")

Lab 4 recap step 3: completed


### Lab 4 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [33]:
print("Lab 4 recap step 4: completed")

Lab 4 recap step 4: completed


### Lab 4 quick recap 5

Pause and summarize one takeaway from the previous cell before moving on.

In [34]:
print("Lab 4 recap step 5: completed")

Lab 4 recap step 5: completed


## Final checkpoint

In [35]:
assert len(X_train_legit) == 792
assert len(X_test) == 200
assert int((y_pred == 1).sum()) == 6
assert abs(precision - 0.3333) < 0.05
assert abs(recall - 1.0) < 0.01
print("Numbers match — you're good.")



Numbers match — you're good.


## Reflection

When is high recall at the cost of precision acceptable?